# 5.3 阶段一：基础极简 FA

从本节起，我们亲手写 FA 算子。第一阶段的目标是**最小可运行**：先不管性能、不管多核、不管掩码——把四拍循环用 Ascend C 跑对，与 golden 对拍通过。

## 1. 阶段规格

| 项 | 规格 |
|--|--|
| 输入 | 只有 Q、K、V 和 scale |
| 输出 | atten |
| 支持范围 | D=128，QS 与 KVS 等长且为 128 的倍数 |
| 排布 | BSND |
| 计算精度 | 非量化（FP16 输入，FP32 中间） |
| 流水 | 串行 |
| 分核 | 单核 |
| mask | 无 |

> 约束越多，代码越简单。QS=KVS 等长、128 对齐意味着没有尾块；单核意味着没有核间切块；串行意味着不用管 buffer 复用。这一阶段只需要专注**四拍循环本身**。

## 2. 工程与 Tiling

按第 4 章 4.3 节的工程结构创建 op_host / op_kernel。本阶段 TilingData 只需最小集：

```cpp
struct FlashAttentionCustomTilingData {
    int32_t bSize;       // batch
    int32_t nSize;       // 头数
    int32_t s1Size;      // QS（=KVS，等长约束）
    int32_t s2Size;      // KVS
    int32_t dSize;       // 128
    float    scaleValue;  // 1/sqrt(D)
};
```

TilingFunc 的职责（对照第 3 章 3.5 节八步流程的**极简版**）：取 shape → 校验等长/对齐约束 → 填表 → `CalcTschBlockDim(1, 1, 1)`（单核）。

## 3. Kernel：四拍循环最简版

单核、串行的 Kernel 主流程：

```cpp
extern "C" __global__ __aicore__ void flash_attention_custom(
    GM_ADDR query, GM_ADDR key, GM_ADDR value, GM_ADDR scale,
    GM_ADDR attentionOut, GM_ADDR workspace, GM_ADDR tiling)
{
    GET_TILING_DATA_WITH_STRUCT(FlashAttentionCustomTilingData, tilingData, tiling);

    // 单核：所有数据直接搬进片上（单 buffer，先算后搬下一段）
    TPipe pipe;
    TQue<TPosition::VECIN, MixQuePosition::VM> queQ, queK, queV;
    pipe.InitBuffer(queQ, BUFFER_NUM, sizeof(half) * S1 * D);   // 单 buffer：BUFFER_NUM=1
    ...

    LocalTensor<half> q = queQ.DeQue<half>();    // Q: [S1, D]
    LocalTensor<half> k = queK.DeQue<half>();    // K: [S2, D]
    LocalTensor<half> v = queV.DeQue<half>();    // V: [S2, D]

    // ---- C1: S = Q · Kᵀ · scale（Cube）----
    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm1, &tilingData.mm1TilingData);
    mm1.SetTensorA(q);
    mm1.SetTensorB(k, true);            // B 转置
    mm1.Iterate();                        // S: [S1, S2] FP32 到 UB

    // ---- V1: P = FlashSoftmax(S)（Vector）----
    // 更新行最大值 m、行指数和 l，得到本块 P
    SoftMaxFlashV2(sTensor, pTensor, m, l, ...);

    // ---- C2: O = P · V（Cube）----
    mm2.SetTensorA(p);                    // P 转回 half
    mm2.SetTensorB(v);
    mm2.Iterate();

    // ---- V2: 末块归一化（Vector）----
    // 等长场景只有一个 KV 块：直接 O /= l
    Div(oTensor, oTensor, l, S1 * D, 1, {1, 1, 8, 8});

    // 写回 GM
    queOut.EnQue(oTensor);
}
```

**要点**：

- 中间量 S/P/O 全部驻留片上（UB / L0），这就是 FA 的本质（第 2 章）；
- FP16 进、FP32 算、FP16 出：S、softmax 中间用 float，P 送 mm2 前转 half；
- 单核串行所以代码最直白——**先把正确性跑通，一切都好办**。

## 4. 调测对拍

按第 4 章 4.5 节四步流程：

```bash
# golden：numpy 按 softmax(QKᵀ·scale)V 生成（FP32 中间过程）
# json：shape [B, S, N, 128]、half、BSND、精度标准

bash run.sh -t compile && bash run.sh -t run     # tiling 单独调
bash run.sh -k compile && bash run.sh -k run     # kernel 单独调
python compare.py dump/ golden/ --rtol 2e-3      # 对拍
```

常见首错排查：

| 症状 | 大概率原因 |
|--|--|
| 数值全 0 | EnQue/DeQue 顺序错，搬运没发生 |
| 每块结果一致但整体错 | offset 算错，所有块读了同一份数据 |
| Softmax 部分错 | m/l 初始化或 FP16/FP32 转换点错 |

## 5. 性能基线

跑通后记下基线：单核串行意味着 Cube 算完干等 Vector、Vector 算完干等 Cube、搬运全程空等。后续每一阶段都在消这些空等——**没有基线就没有优化感**。

## 小测验

1. 阶段一为什么规定「QS 与 KVS 等长且为 128 的倍数」？这个约束帮我们省掉了什么？
2. 四拍循环中哪两拍跑在 Cube、哪两拍跑在 Vector？P 矩阵送 mm2 前要做什么转换？
3. 单核串行版本里，「搬运」和「计算」的时间关系是什么？